In [1]:

# ============================================================
# Stage 09: Error Analysis for Emotion Classification
#
# Model:
#   BERT-base-uncased
#
# Checkpoint:
#   Stage 08 baseline model
#
# Data:
#   results/splits/validation.csv
#
# Outputs:
#   1. validation_predictions.csv
#   2. misclassified_examples.csv
#   3. high_confidence_errors.csv
#   4. low_confidence_errors.csv
#   5. classification_report.csv
#   6. confusion_matrix.csv
#   7. confusion_matrix_normalized.csv
#   8. class_metrics.csv
#   9. confusion_pairs.csv
#  10. confidence_analysis.csv
#  11. PNG visualizations
#
# IMPORTANT:
#   - Do not load the test set in this script.
#   - Use validation data for error analysis only.
# ============================================================

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
)

from safetensors.torch import load_file

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
)


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

MODEL_NAME = "bert-base-uncased"
NUM_CLASSES = 6
MAX_LEN = 128
BATCH_SIZE = 32

VALIDATION_PATH = Path("results/splits/validation.csv")

CHECKPOINT_PATH = Path(
    "results/augmentation/wordnet_comparison/"
    "baseline/best_model.safetensors"
)

OUTPUT_DIR = Path("results/error_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_AMP = torch.cuda.is_available()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Replace these names only after verifying the real label mapping.
# Numeric labels remain the source of truth.
CLASS_NAMES = {
    0: "label_0",
    1: "label_1",
    2: "label_2",
    3: "label_3",
    4: "label_4",
    5: "label_5",
}

# Number of examples to save per confusion pair.
EXAMPLES_PER_PAIR = 100

# High-confidence errors: prediction confidence >= threshold.
HIGH_CONFIDENCE_THRESHOLD = 0.90

# Low-confidence errors: prediction confidence < threshold.
LOW_CONFIDENCE_THRESHOLD = 0.60


# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

print("=" * 70)
print("Stage 09: Error Analysis")
print("=" * 70)
print(f"Device: {DEVICE}")
print(f"AMP enabled: {USE_AMP}")
print(f"Validation file: {VALIDATION_PATH.resolve()}")
print(f"Checkpoint: {CHECKPOINT_PATH.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# ============================================================
# 3. Check Files
# ============================================================

if not VALIDATION_PATH.exists():
    raise FileNotFoundError(
        f"Validation data not found: {VALIDATION_PATH.resolve()}"
    )

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH.resolve()}"
    )


# ============================================================
# 4. Load Validation Data
# ============================================================

val_df = pd.read_csv(VALIDATION_PATH)

required_columns = {"text", "label"}

if not required_columns.issubset(val_df.columns):
    raise ValueError(
        f"Validation CSV must contain {required_columns}"
    )

val_df = val_df[["text", "label"]].copy()

val_df["text"] = val_df["text"].fillna("").astype(str)
val_df["label"] = pd.to_numeric(
    val_df["label"], errors="raise"
).astype(int)

# Match the text-cleaning rule used in the training script.
val_df = val_df[
    val_df["text"].str.strip().ne("")
].reset_index(drop=True)

invalid_labels = sorted(
    set(val_df["label"].unique()) - set(range(NUM_CLASSES))
)

if invalid_labels:
    raise ValueError(
        f"Invalid labels found: {invalid_labels}"
    )

print("\n[Validation Dataset]")
print(f"Number of rows: {len(val_df):,}")

print("\nActual class counts:")
print(
    val_df["label"]
    .value_counts()
    .reindex(range(NUM_CLASSES), fill_value=0)
    .sort_index()
    .to_string()
)


# ============================================================
# 5. Dataset
# ============================================================

class EmotionDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": self.labels[idx],
        }


# ============================================================
# 6. Load Tokenizer and Model
# ============================================================

print("\n[Loading Tokenizer]")

tokenizer = BertTokenizerFast.from_pretrained(
    MODEL_NAME
)


def collate_fn(batch):
    texts = [item["text"] for item in batch]

    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long,
    )

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    encoded["labels"] = labels

    return encoded


val_loader = DataLoader(
    EmotionDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_fn,
)

print("\n[Loading Model]")

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
)

# The checkpoint was saved with safetensors.save_file().
state_dict = load_file(
    str(CHECKPOINT_PATH),
    device="cpu",
)

model.load_state_dict(
    state_dict,
    strict=True,
)

model.to(DEVICE)
model.eval()

print("Checkpoint loaded successfully.")


# ============================================================
# 7. Predict on Validation Data
# ============================================================

all_texts = []
all_true_labels = []
all_predicted_labels = []
all_probabilities = []

print("\n[Running Validation Predictions]")

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        texts = batch.pop("labels")
        labels = texts.to(DEVICE)

        model_inputs = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            outputs = model(**model_inputs)
            logits = outputs.logits

        probabilities = torch.softmax(
            logits.float(),
            dim=-1,
        )

        predictions = probabilities.argmax(dim=-1)

        all_true_labels.extend(
            labels.cpu().numpy().tolist()
        )

        all_predicted_labels.extend(
            predictions.cpu().numpy().tolist()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        all_texts.extend(
            val_df.iloc[
                len(all_texts):
                len(all_texts) + len(labels)
            ]["text"].tolist()
        )

        if (
            (batch_idx + 1) % 100 == 0
            or (batch_idx + 1) == len(val_loader)
        ):
            print(
                f"Processed "
                f"{min((batch_idx + 1) * BATCH_SIZE, len(val_df)):,}"
                f" / {len(val_df):,}"
            )


# ============================================================
# 8. Build Prediction DataFrame
# ============================================================

y_true = np.asarray(all_true_labels, dtype=int)
y_pred = np.asarray(all_predicted_labels, dtype=int)
probabilities = np.asarray(
    all_probabilities,
    dtype=np.float32,
)

if len(y_true) != len(val_df):
    raise RuntimeError(
        "Prediction row count does not match validation data."
    )

if not np.isfinite(probabilities).all():
    raise RuntimeError(
        "Non-finite prediction probabilities detected."
    )

predicted_confidence = probabilities.max(axis=1)

sorted_probs = np.sort(probabilities, axis=1)

top1_prob = sorted_probs[:, -1]
top2_prob = sorted_probs[:, -2]

confidence_margin = top1_prob - top2_prob

# Predictive entropy. Higher entropy means the probability
# distribution is more spread out across classes.
entropy = -np.sum(
    probabilities * np.log(
        np.clip(probabilities, 1e-12, 1.0)
    ),
    axis=1,
)

results_df = pd.DataFrame({
    "text": all_texts,
    "true_label": y_true,
    "true_class": [
        CLASS_NAMES[int(x)] for x in y_true
    ],
    "predicted_label": y_pred,
    "predicted_class": [
        CLASS_NAMES[int(x)] for x in y_pred
    ],
    "correct": y_true == y_pred,
    "confidence": predicted_confidence,
    "top2_margin": confidence_margin,
    "prediction_entropy": entropy,
})

for label in range(NUM_CLASSES):
    results_df[f"prob_label_{label}"] = probabilities[:, label]

results_path = OUTPUT_DIR / "validation_predictions.csv"

results_df.to_csv(
    results_path,
    index=False,
)

print(f"\nSaved predictions: {results_path.resolve()}")


# ============================================================
# 9. Overall Metrics
# ============================================================

accuracy = accuracy_score(y_true, y_pred)

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0,
    )
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    average="weighted",
    zero_division=0,
)

overall_metrics = {
    "validation_samples": int(len(y_true)),
    "accuracy": float(accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "correct_predictions": int((y_true == y_pred).sum()),
    "incorrect_predictions": int((y_true != y_pred).sum()),
    "error_rate": float((y_true != y_pred).mean()),
    "mean_prediction_confidence": float(
        predicted_confidence.mean()
    ),
}

with open(
    OUTPUT_DIR / "overall_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        overall_metrics,
        f,
        ensure_ascii=False,
        indent=4,
    )

print("\n[Overall Metrics]")
for key, value in overall_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value:,}")


# ============================================================
# 10. Classification Report
# ============================================================

report_dict = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=[
        CLASS_NAMES[i] for i in range(NUM_CLASSES)
    ],
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report_dict).transpose()

report_df.to_csv(
    OUTPUT_DIR / "classification_report.csv"
)

print("\n[Classification Report]")
print(
    report_df.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 11. Class-Level Metrics
# ============================================================

precision, recall, f1, support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        average=None,
        zero_division=0,
    )
)

class_metrics = pd.DataFrame({
    "label": list(range(NUM_CLASSES)),
    "class_name": [
        CLASS_NAMES[i] for i in range(NUM_CLASSES)
    ],
    "support": support,
    "precision": precision,
    "recall": recall,
    "f1": f1,
})

class_metrics["correct_count"] = [
    int(((y_true == i) & (y_pred == i)).sum())
    for i in range(NUM_CLASSES)
]

class_metrics["error_count"] = [
    int(((y_true == i) & (y_pred != i)).sum())
    for i in range(NUM_CLASSES)
]

class_metrics["error_rate"] = (
    class_metrics["error_count"]
    / class_metrics["support"].replace(0, np.nan)
)

class_metrics.to_csv(
    OUTPUT_DIR / "class_metrics.csv",
    index=False,
)

print("\n[Class-Level Metrics]")
print(
    class_metrics.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ============================================================
# 12. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"actual_{CLASS_NAMES[i]}"
        for i in range(NUM_CLASSES)
    ],
    columns=[
        f"predicted_{CLASS_NAMES[i]}"
        for i in range(NUM_CLASSES)
    ],
)

cm_df.to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

# Row-normalized matrix:
# each row represents the distribution of predictions
# for a particular true class.
row_sums = cm.sum(axis=1, keepdims=True)

cm_normalized = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0,
)

cm_normalized_df = pd.DataFrame(
    cm_normalized,
    index=[
        f"actual_{CLASS_NAMES[i]}"
        for i in range(NUM_CLASSES)
    ],
    columns=[
        f"predicted_{CLASS_NAMES[i]}"
        for i in range(NUM_CLASSES)
    ],
)

cm_normalized_df.to_csv(
    OUTPUT_DIR / "confusion_matrix_normalized.csv"
)

print("\n[Confusion Matrix]")
print(cm_df.to_string())


# ============================================================
# 13. Confusion Matrix Visualizations
# ============================================================

def plot_confusion_matrix(
    matrix,
    labels,
    title,
    output_path,
    normalize=False,
):
    fig, ax = plt.subplots(figsize=(9, 7))

    image = ax.imshow(
        matrix,
        interpolation="nearest",
        aspect="auto",
    )

    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted Label",
        ylabel="True Label",
        title=title,
    )

    plt.setp(
        ax.get_xticklabels(),
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    threshold = (
        np.nanmax(matrix) / 2
        if matrix.size
        else 0
    )

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if normalize:
                text_value = f"{matrix[i, j]:.2f}"
            else:
                text_value = f"{int(matrix[i, j])}"

            ax.text(
                j,
                i,
                text_value,
                ha="center",
                va="center",
                color=(
                    "white"
                    if matrix[i, j] > threshold
                    else "black"
                ),
                fontsize=9,
            )

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(fig)


class_labels = [
    CLASS_NAMES[i] for i in range(NUM_CLASSES)
]

plot_confusion_matrix(
    cm,
    class_labels,
    "Validation Confusion Matrix",
    OUTPUT_DIR / "confusion_matrix.png",
)

plot_confusion_matrix(
    cm_normalized,
    class_labels,
    "Normalized Validation Confusion Matrix",
    OUTPUT_DIR / "confusion_matrix_normalized.png",
    normalize=True,
)


# ============================================================
# 14. Class-Level F1 Visualization
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    class_metrics["class_name"],
    class_metrics["f1"],
)

ax.set_title("Validation F1 by Class")
ax.set_xlabel("Class")
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1)

for index, value in enumerate(class_metrics["f1"]):
    ax.text(
        index,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        fontsize=9,
    )

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "class_f1.png",
    dpi=180,
    bbox_inches="tight",
)

plt.close(fig)


# ============================================================
# 15. Misclassified Examples
# ============================================================

errors_df = results_df[
    results_df["correct"] == False
].copy()

errors_df = errors_df.sort_values(
    "confidence",
    ascending=False,
)

errors_df.to_csv(
    OUTPUT_DIR / "misclassified_examples.csv",
    index=False,
)

print("\n[Misclassification Summary]")
print(f"Total errors: {len(errors_df):,}")

print("\nMost common true-label -> predicted-label errors:")

error_pairs = (
    errors_df
    .groupby(
        ["true_label", "predicted_label"],
        as_index=False,
    )
    .agg(
        error_count=("text", "size"),
        mean_confidence=("confidence", "mean"),
    )
    .sort_values(
        "error_count",
        ascending=False,
    )
)

error_pairs["true_class"] = error_pairs[
    "true_label"
].map(CLASS_NAMES)

error_pairs["predicted_class"] = error_pairs[
    "predicted_label"
].map(CLASS_NAMES)

error_pairs.to_csv(
    OUTPUT_DIR / "confusion_pairs.csv",
    index=False,
)

print(
    error_pairs[
        [
            "true_class",
            "predicted_class",
            "error_count",
            "mean_confidence",
        ]
    ]
    .head(20)
    .to_string(index=False)
)


# ============================================================
# 16. High-Confidence and Low-Confidence Errors
# ============================================================

high_confidence_errors = errors_df[
    errors_df["confidence"] >= HIGH_CONFIDENCE_THRESHOLD
].copy()

low_confidence_errors = errors_df[
    errors_df["confidence"] < LOW_CONFIDENCE_THRESHOLD
].copy()

high_confidence_errors.to_csv(
    OUTPUT_DIR / "high_confidence_errors.csv",
    index=False,
)

low_confidence_errors.to_csv(
    OUTPUT_DIR / "low_confidence_errors.csv",
    index=False,
)

print("\n[Confidence-Based Error Analysis]")
print(
    f"High-confidence errors "
    f"(confidence >= {HIGH_CONFIDENCE_THRESHOLD}): "
    f"{len(high_confidence_errors):,}"
)

print(
    f"Low-confidence errors "
    f"(confidence < {LOW_CONFIDENCE_THRESHOLD}): "
    f"{len(low_confidence_errors):,}"
)


# ============================================================
# 17. Confidence Analysis
# ============================================================

results_df["confidence_bin"] = pd.cut(
    results_df["confidence"],
    bins=[0.0, 0.50, 0.60, 0.70, 0.80, 0.90, 1.000001],
    labels=[
        "0.00-0.50",
        "0.50-0.60",
        "0.60-0.70",
        "0.70-0.80",
        "0.80-0.90",
        "0.90-1.00",
    ],
    include_lowest=True,
)

confidence_analysis = (
    results_df
    .groupby(
        "confidence_bin",
        observed=False,
    )
    .agg(
        sample_count=("correct", "size"),
        accuracy=("correct", "mean"),
        mean_confidence=("confidence", "mean"),
        error_count=("correct", lambda x: int((~x).sum())),
    )
    .reset_index()
)

confidence_analysis.to_csv(
    OUTPUT_DIR / "confidence_analysis.csv",
    index=False,
)

print("\n[Confidence Analysis]")
print(
    confidence_analysis.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


# ============================================================
# 18. Save Representative Errors for Each Confusion Pair
# ============================================================

pair_examples_dir = OUTPUT_DIR / "pair_examples"
pair_examples_dir.mkdir(parents=True, exist_ok=True)

for row in error_pairs.itertuples(index=False):
    true_label = int(row.true_label)
    predicted_label = int(row.predicted_label)

    pair_df = errors_df[
        (errors_df["true_label"] == true_label)
        & (errors_df["predicted_label"] == predicted_label)
    ].copy()

    # Select the highest-confidence mistakes first.
    pair_df = pair_df.head(EXAMPLES_PER_PAIR)

    output_name = (
        f"true_{true_label}_predicted_{predicted_label}.csv"
    )

    pair_df.to_csv(
        pair_examples_dir / output_name,
        index=False,
    )


# ============================================================
# 19. Automatically Generate an Initial Summary
# ============================================================

worst_class = class_metrics.sort_values(
    "f1",
    ascending=True,
).iloc[0]

best_class = class_metrics.sort_values(
    "f1",
    ascending=False,
).iloc[0]

most_common_error = (
    error_pairs.iloc[0].to_dict()
    if len(error_pairs) > 0
    else None
)

summary = {
    "overall_metrics": overall_metrics,
    "lowest_f1_class": {
        "label": int(worst_class["label"]),
        "class_name": str(worst_class["class_name"]),
        "f1": float(worst_class["f1"]),
        "recall": float(worst_class["recall"]),
        "error_count": int(worst_class["error_count"]),
    },
    "highest_f1_class": {
        "label": int(best_class["label"]),
        "class_name": str(best_class["class_name"]),
        "f1": float(best_class["f1"]),
        "recall": float(best_class["recall"]),
        "error_count": int(best_class["error_count"]),
    },
    "most_common_error_pair": (
        {
            "true_label": int(most_common_error["true_label"]),
            "predicted_label": int(
                most_common_error["predicted_label"]
            ),
            "error_count": int(
                most_common_error["error_count"]
            ),
            "mean_confidence": float(
                most_common_error["mean_confidence"]
            ),
        }
        if most_common_error is not None
        else None
    ),
    "high_confidence_error_count": int(
        len(high_confidence_errors)
    ),
    "low_confidence_error_count": int(
        len(low_confidence_errors)
    ),
}

with open(
    OUTPUT_DIR / "error_analysis_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=4,
    )

print("\n" + "=" * 70)
print("Stage 09 Completed")
print("=" * 70)

print("\nLowest-F1 class:")
print(json.dumps(
    summary["lowest_f1_class"],
    ensure_ascii=False,
    indent=2,
))

print("\nHighest-F1 class:")
print(json.dumps(
    summary["highest_f1_class"],
    ensure_ascii=False,
    indent=2,
))

print("\nMost common error pair:")
print(json.dumps(
    summary["most_common_error_pair"],
    ensure_ascii=False,
    indent=2,
))

print("\nAll results saved to:")
print(OUTPUT_DIR.resolve())


Stage 09: Error Analysis
Device: cuda
AMP enabled: True
Validation file: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/splits/validation.csv
Checkpoint: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/augmentation/wordnet_comparison/baseline/best_model.safetensors
Output directory: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/error_analysis
GPU: NVIDIA GeForce RTX 5070 Ti

[Validation Dataset]
Number of rows: 65,422

Actual class counts:
label
0    18986
1    22211
2     5377
3     9013
4     7505
5     2330

[Loading Tokenizer]

[Loading Model]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint loaded successfully.

[Running Validation Predictions]
Processed 3,200 / 65,422
Processed 6,400 / 65,422
Processed 9,600 / 65,422
Processed 12,800 / 65,422
Processed 16,000 / 65,422
Processed 19,200 / 65,422
Processed 22,400 / 65,422
Processed 25,600 / 65,422
Processed 28,800 / 65,422
Processed 32,000 / 65,422
Processed 35,200 / 65,422
Processed 38,400 / 65,422
Processed 41,600 / 65,422
Processed 44,800 / 65,422
Processed 48,000 / 65,422
Processed 51,200 / 65,422
Processed 54,400 / 65,422
Processed 57,600 / 65,422
Processed 60,800 / 65,422
Processed 64,000 / 65,422
Processed 65,422 / 65,422

Saved predictions: /mnt/e/university/grade4_semester2/딥러닝개론/팀프로젝트/Emotion_Analysis/results/error_analysis/validation_predictions.csv

[Overall Metrics]
validation_samples: 65,422
accuracy: 0.942970
macro_precision: 0.892082
macro_recall: 0.956599
macro_f1: 0.918226
weighted_f1: 0.944823
correct_predictions: 61,691
incorrect_predictions: 3,731
error_rate: 0.057030
mean_prediction_confiden